# 15. Optimización y selección de modelos finales

## 15.1. Objetivo

En los notebooks anteriores se han entrenado y comparado diferentes algoritmos de clasificación para predecir los códigos de calidad de las tres componentes de irradiancia: GHI, DNI y DHI.

El objetivo de este notebook es seleccionar los modelos con mejor rendimiento obtenidos previamente y optimizar sus hiperparámetros para obtener una configuración final para cada variable objetivo.

La optimización se realizará utilizando exclusivamente los datos correspondientes a 2024. Los datos de 2023 permanecerán completamente aislados durante esta fase y se utilizarán únicamente al final del proceso para realizar una evaluación interanual independiente de los modelos optimizados.

La métrica principal utilizada para seleccionar las configuraciones será Macro F1, debido al desequilibrio existente entre las clases y al interés en evaluar de forma equilibrada el rendimiento sobre códigos mayoritarios y minoritarios. Como métricas complementarias se utilizarán Balanced Accuracy y Weighted F1.

El flujo seguido será:

1. Recuperación de los mejores modelos obtenidos en los notebooks anteriores.
2. Definición de los espacios de hiperparámetros.
3. Optimización utilizando únicamente los datos de entrenamiento de 2024.
4. Selección de la mejor configuración para cada target.
5. Evaluación final sobre los datos independientes de 2023.
6. Comparación entre modelos originales y optimizados.
7. Persistencia de los modelos y resultados finales.

In [7]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report
)

RANDOM_STATE = 42

TRAIN_YEAR = 2024
TEST_YEAR = 2023

TARGETS = [
    "codigo_ghi",
    "codigo_dni",
    "codigo_dhi"
]

PRIMARY_METRIC = "f1_macro"

DATA_PATH = Path(
    "../data/processed/dataset_solar_2023_2024_v3.parquet"
)

MODEL_OUTPUT_PATH = Path(
    "../models/final"
)

MODEL_OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

In [8]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [9]:
df = pd.read_parquet(DATA_PATH)

print(f"Dataset completo: {df.shape}")
print(
    f"Periodo: {df['fecha'].min()} "
    f"→ {df['fecha'].max()}"
)

Dataset completo: (1052640, 27)
Periodo: 2023-01-01 00:00:00 → 2024-12-31 23:59:00


In [10]:
train_df = (
    df[df["ano"] == TRAIN_YEAR]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    df[df["ano"] == TEST_YEAR]
    .copy()
    .reset_index(drop=True)
)

print(
    f"Train ({TRAIN_YEAR}): "
    f"{train_df.shape}"
)

print(
    f"Test ({TEST_YEAR}): "
    f"{test_df.shape}"
)

Train (2024): (527040, 27)
Test (2023): (525600, 27)


In [11]:
EXCLUDED_COLUMNS = [
    "fecha",
    "ano",
    "codigo_ghi",
    "codigo_dni",
    "codigo_dhi"
]

CANDIDATE_FEATURES = [
    column
    for column in df.columns
    if column not in EXCLUDED_COLUMNS
]

print(
    f"Variables candidatas disponibles: "
    f"{len(CANDIDATE_FEATURES)}"
)

CANDIDATE_FEATURES

Variables candidatas disponibles: 22


['mes_sin',
 'mes_cos',
 'dia',
 'hora_sin',
 'hora_cos',
 'minuto',
 'ghi',
 'dni',
 'dhi',
 'ghi_estimado',
 'irr_null',
 'error_balance',
 'error_balance_abs',
 'error_balance_rel',
 'elevacion_solar',
 'periodo_solar',
 'temperatura',
 'velocidad_viento',
 'humedad_relativa',
 'direccion_viento_sin',
 'direccion_viento_cos',
 'var_meteo_imp']

In [12]:
from src.database.postgresql_persistence import get_postgresql_connection

In [19]:
query_all_results = """
SELECT
    m.model_id,
    m.model_name,
    m.target,
    m.train_year,
    m.test_year,
    m.n_features,
    m.features,
    m.hyperparameters,
    r.f1_macro,
    r.balanced_accuracy,
    r.f1_weighted,
    r.created_at
FROM solar.models AS m
INNER JOIN solar.results AS r
    ON m.model_id = r.model_id
ORDER BY
    m.target,
    r.f1_macro DESC;
"""

conn = get_postgresql_connection()

try:
    all_results = pd.read_sql_query(
        query_all_results,
        conn
    )
finally:
    conn.close()

print(
    f"Número total de experimentos registrados: "
    f"{len(all_results)}"
)

all_results.head()

Número total de experimentos registrados: 18


C:\Users\zacar\AppData\Local\Temp\ipykernel_15336\3418399148.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  all_results = pd.read_sql_query(


,model_id,model_name,target,train_year,test_year,n_features,features,hyperparameters,f1_macro,balanced_accuracy,f1_weighted,created_at
0,3,HistGradientBoostingClassifier,codigo_dhi,2024,2023,22,"[mes_sin, mes_cos, dia, hora_sin, hora_cos, mi...","{'tol': 1e-07, 'loss': 'log_loss', 'scoring': ...",0.498600,0.572575,0.882477,2026-08-13 18:41:04.711730
1,21,HistGradientBoosting,codigo_dhi,2024,2023,18,"[mes_sin, mes_cos, dia, hora_sin, hora_cos, mi...","{'tol': 1e-07, 'loss': 'log_loss', 'scoring': ...",0.478681,0.555883,0.863185,2026-08-18 08:11:53.092781
2,24,MLP,codigo_dhi,2024,2023,14,"[mes_sin, mes_cos, dia, hora_sin, hora_cos, mi...","{'optimizer': 'Adam', 'activation': 'relu', 'b...",0.451350,0.515467,0.842825,2026-08-19 08:44:36.752629
3,18,RandomForest,codigo_dhi,2024,2023,14,"[mes_sin, mes_cos, dia, hora_sin, hora_cos, mi...","{'n_jobs': -1, 'verbose': 0, 'bootstrap': True...",0.450474,0.505040,0.847037,2026-08-18 08:11:42.893393
4,15,LogisticRegression,codigo_dhi,2024,2023,14,"[mes_sin, mes_cos, dia, hora_sin, hora_cos, mi...","{'C': 1.0, 'tol': 0.0001, 'dual': False, 'n_jo...",0.413080,0.560074,0.778568,2026-08-18 08:11:24.517424


In [20]:
results_summary = all_results[
    [
        "model_id",
        "target",
        "model_name",
        "n_features",
        "f1_macro",
        "balanced_accuracy",
        "f1_weighted"
    ]
].copy()

results_summary

,model_id,target,model_name,n_features,f1_macro,balanced_accuracy,f1_weighted
0,3,codigo_dhi,HistGradientBoostingClassifier,22,0.498600,0.572575,0.882477
1,21,codigo_dhi,HistGradientBoosting,18,0.478681,0.555883,0.863185
2,24,codigo_dhi,MLP,14,0.451350,0.515467,0.842825
3,18,codigo_dhi,RandomForest,14,0.450474,0.505040,0.847037
4,15,codigo_dhi,LogisticRegression,14,0.413080,0.560074,0.778568
5,12,codigo_dhi,DummyClassifier,0,0.313993,0.333333,0.838667
6,20,codigo_dni,HistGradientBoosting,18,0.534738,0.625648,0.940909
7,17,codigo_dni,RandomForest,14,0.489022,0.551082,0.927517
8,2,codigo_dni,HistGradientBoostingClassifier,22,0.486994,0.616805,0.932485
9,23,codigo_dni,MLP,14,0.462769,0.675691,0.904146


In [21]:
best_results = (
    all_results
    .sort_values(
        by=[
            "target",
            "f1_macro",
            "balanced_accuracy",
            "f1_weighted"
        ],
        ascending=[
            True,
            False,
            False,
            False
        ]
    )
    .drop_duplicates(
        subset="target",
        keep="first"
    )
    .reset_index(drop=True)
)

best_results[
    [
        "model_id",
        "target",
        "model_name",
        "n_features",
        "f1_macro",
        "balanced_accuracy",
        "f1_weighted"
    ]
]

,model_id,target,model_name,n_features,f1_macro,balanced_accuracy,f1_weighted
0,3,codigo_dhi,HistGradientBoostingClassifier,22,0.498600,0.572575,0.882477
1,20,codigo_dni,HistGradientBoosting,18,0.534738,0.625648,0.940909
2,22,codigo_ghi,MLP,14,0.743533,0.782470,0.883894


In [22]:
best_candidates = {
    "codigo_ghi": [
        "MLP",
        "LogisticRegression"
    ],
    "codigo_dni": [
        "HistGradientBoosting"
    ],
    "codigo_dhi": [
        "HistGradientBoostingClassifier"
    ]
}

In [23]:
candidates_to_optimize = pd.concat([
    
    # GHI: dos mejores candidatos
    (
        all_results[
            all_results["target"] == "codigo_ghi"
        ]
        .sort_values("f1_macro", ascending=False)
        .head(2)
    ),
    
    # DNI: mejor candidato
    (
        all_results[
            all_results["target"] == "codigo_dni"
        ]
        .sort_values("f1_macro", ascending=False)
        .head(1)
    ),
    
    # DHI: mejor candidato
    (
        all_results[
            all_results["target"] == "codigo_dhi"
        ]
        .sort_values("f1_macro", ascending=False)
        .head(1)
    )
    
]).reset_index(drop=True)

candidates_to_optimize[
    [
        "model_id",
        "target",
        "model_name",
        "n_features",
        "f1_macro",
        "balanced_accuracy",
        "f1_weighted"
    ]
]

,model_id,target,model_name,n_features,f1_macro,balanced_accuracy,f1_weighted
0,22,codigo_ghi,MLP,14,0.743533,0.782470,0.883894
1,13,codigo_ghi,LogisticRegression,14,0.646299,0.810603,0.790116
2,20,codigo_dni,HistGradientBoosting,18,0.534738,0.625648,0.940909
3,3,codigo_dhi,HistGradientBoostingClassifier,22,0.498600,0.572575,0.882477


In [28]:
train_df["mes_validation"] = train_df["fecha"].dt.month

monthly_target_distribution = {}

for target in TARGETS:
    monthly_distribution = pd.crosstab(
        train_df["mes_validation"],
        train_df[target],
        normalize="index"
    )

    monthly_target_distribution[target] = monthly_distribution

    print(f"\n{'=' * 70}")
    print(f"Distribución mensual de clases — {target}")
    print(f"{'=' * 70}")
    
    display(monthly_distribution.round(4))


Distribución mensual de clases — codigo_ghi


codigo_ghi,0,1
mes_validation,,
1,0.9979,0.0021
2,0.8465,0.1535
3,0.5476,0.4524
4,0.8638,0.1362
5,0.9026,0.0974
6,0.9429,0.0571
7,0.7765,0.2235
8,0.9800,0.0200
9,0.4329,0.5671



Distribución mensual de clases — codigo_dni


codigo_dni,0,1,2
mes_validation,,,
1,0.9959,0.0000,0.0041
2,0.8437,0.1552,0.0011
3,0.5471,0.4528,0.0002
4,0.8571,0.1385,0.0044
5,0.9004,0.0978,0.0019
6,0.9642,0.0337,0.0020
7,0.8052,0.1946,0.0002
8,0.9974,0.0015,0.0012
9,0.4329,0.5671,0.0000



Distribución mensual de clases — codigo_dhi


codigo_dhi,0,1,2
mes_validation,,,
1,0.9904,0.0096,0.0000
2,0.8415,0.1585,0.0000
3,0.5465,0.4524,0.0011
4,0.8539,0.1461,0.0000
5,0.7405,0.2595,0.0000
6,0.5669,0.4331,0.0000
7,0.7316,0.2649,0.0035
8,0.9080,0.0920,0.0000
9,0.4329,0.5671,0.0000


In [29]:
class_presence_by_month = []

for target in TARGETS:
    for month in sorted(train_df["mes_validation"].unique()):
        month_data = train_df.loc[
            train_df["mes_validation"] == month,
            target
        ]

        counts = month_data.value_counts()

        class_presence_by_month.append({
            "target": target,
            "mes_validation": month,
            "n_samples": len(month_data),
            "n_classes": month_data.nunique(),
            "classes": sorted(month_data.unique().tolist()),
            "min_class_count": counts.min()
        })

class_presence_by_month = pd.DataFrame(
    class_presence_by_month
)

class_presence_by_month

,target,mes_validation,n_samples,n_classes,classes,min_class_count
0,codigo_ghi,1,44640,2,"[0, 1]",94
1,codigo_ghi,2,41760,2,"[0, 1]",6409
2,codigo_ghi,3,44640,2,"[0, 1]",20194
3,codigo_ghi,4,43200,2,"[0, 1]",5885
4,codigo_ghi,5,44640,2,"[0, 1]",4350
5,codigo_ghi,6,43200,2,"[0, 1]",2468
6,codigo_ghi,7,44640,2,"[0, 1]",9976
7,codigo_ghi,8,44640,2,"[0, 1]",892
8,codigo_ghi,9,43200,2,"[0, 1]",18702
9,codigo_ghi,10,44640,2,"[0, 1]",1752
